In [1]:
import pandas as pd

df = pd.read_csv("../dataset/processed/cleaned_tickets.csv")

df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")

df["ticket_text"] = df["subject"] + " " + df["body"]

X = df["ticket_text"]
y = df["queue"]

print("Shape:", df.shape)

print("\nQueue distribution:")
print(y.value_counts())


Shape: (20000, 18)

Queue distribution:
queue
Technical Support                  5824
Product Support                    3708
Customer Service                   3152
IT Support                         2292
Billing and Payments               2086
Returns and Exchanges              1001
Service Outages and Maintenance     764
Sales and Pre-Sales                 572
Human Resources                     338
General Inquiry                     263
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (16000,)
Testing: (4000,)


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    strip_accents="unicode"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (16000, 30000)
Testing TF-IDF shape: (4000, 30000)


In [4]:
from sklearn.svm import LinearSVC

queue_model = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=10000
)

queue_model.fit(X_train_tfidf, y_train)

y_pred = queue_model.predict(X_test_tfidf)

print("✅ Queue model trained successfully!")

✅ Queue model trained successfully!


In [5]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print("Queue Model Accuracy:", accuracy)
print("Queue Model Macro F1:", macro_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Queue Model Accuracy: 0.499
Queue Model Macro F1: 0.4829991224408546

Classification Report:
                                 precision    recall  f1-score   support

           Billing and Payments       0.72      0.79      0.75       417
               Customer Service       0.43      0.44      0.44       630
                General Inquiry       0.46      0.36      0.40        53
                Human Resources       0.52      0.49      0.50        68
                     IT Support       0.40      0.43      0.42       458
                Product Support       0.47      0.41      0.44       742
          Returns and Exchanges       0.38      0.39      0.38       200
            Sales and Pre-Sales       0.39      0.51      0.44       114
Service Outages and Maintenance       0.49      0.54      0.51       153
              Technical Support       0.55      0.53      0.54      1165

                       accuracy                           0.50      4000
                      macro a

In [6]:
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

encoder = OneHotEncoder(handle_unknown="ignore")

X_train_meta = encoder.fit_transform(
    X_train.to_frame().assign(
        type=df.loc[X_train.index, "type"].values,
        language=df.loc[X_train.index, "language"].values
    )[["type", "language"]]
)

X_test_meta = encoder.transform(
    X_test.to_frame().assign(
        type=df.loc[X_test.index, "type"].values,
        language=df.loc[X_test.index, "language"].values
    )[["type", "language"]]
)

X_train_combined = hstack([
    X_train_tfidf,
    X_train_meta
]).tocsr()

X_test_combined = hstack([
    X_test_tfidf,
    X_test_meta
]).tocsr()

print("Training shape:", X_train_combined.shape)
print("Testing shape:", X_test_combined.shape)

Training shape: (16000, 30006)
Testing shape: (4000, 30006)


In [7]:
queue_model_meta = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=10000
)

queue_model_meta.fit(
    X_train_combined,
    y_train
)

y_pred_meta = queue_model_meta.predict(
    X_test_combined
)

accuracy_meta = accuracy_score(
    y_test,
    y_pred_meta
)

macro_f1_meta = f1_score(
    y_test,
    y_pred_meta,
    average="macro"
)

print("Improved Queue Accuracy:", accuracy_meta)
print("Improved Queue Macro F1:", macro_f1_meta)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_meta))

Improved Queue Accuracy: 0.501
Improved Queue Macro F1: 0.48269631151252124

Classification Report:
                                 precision    recall  f1-score   support

           Billing and Payments       0.73      0.79      0.76       417
               Customer Service       0.45      0.46      0.45       630
                General Inquiry       0.44      0.38      0.41        53
                Human Resources       0.54      0.50      0.52        68
                     IT Support       0.39      0.42      0.41       458
                Product Support       0.47      0.41      0.44       742
          Returns and Exchanges       0.37      0.37      0.37       200
            Sales and Pre-Sales       0.38      0.47      0.42       114
Service Outages and Maintenance       0.48      0.56      0.52       153
              Technical Support       0.55      0.53      0.54      1165

                       accuracy                           0.50      4000
                      

In [8]:
import os
import joblib

os.makedirs("../models/queue", exist_ok=True)

joblib.dump(
    queue_model,
    "../models/queue/queue_model.pkl"
)

joblib.dump(
    tfidf,
    "../models/queue/tfidf_vectorizer.pkl"
)

print("✅ Queue model saved!")

✅ Queue model saved!
